In [0]:
# ==============================================================================
# PROJETO 2 - FASE 1: INGESTÃO VIA API COINGECKO (CAMADA BRONZE - SCHEMA FORÇADO)
# ==============================================================================
import requests
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Requisição HTTP para a API da CoinGecko
url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "ids": "bitcoin,ethereum,solana",
    "order": "market_cap_desc",
    "sparkline": "false"
}

print("Iniciando a coleta de dados da API CoinGecko...")
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    
    # 2. Definindo o Schema explícito para evitar conflitos entre LongType e DoubleType
    # Mapeamos apenas os campos principais que queremos garantir a tipagem correta
    custom_schema = StructType([
        StructField("id", StringType(), True),
        StructField("symbol", StringType(), True),
        StructField("name", StringType(), True),
        StructField("current_price", DoubleType(), True),      # Força Double para evitar o erro de DoubleType
        StructField("market_cap", DoubleType(), True),         # Força Double
        StructField("total_volume", DoubleType(), True),       # Força Double
        StructField("price_change_percentage_24h", DoubleType(), True),
        StructField("last_updated", StringType(), True)
    ])
    
    # 3. Criando o DataFrame passando a lista de dados e o nosso schema customizado
    df_raw = spark.createDataFrame(data, schema=custom_schema)
    
    # 4. Adiciona a coluna de metadado (Data e Hora do momento exato da coleta)
    df_bronze = df_raw.withColumn("coletado_em", current_timestamp())
    
    # 5. Salva na Camada Bronze com a estratégia APPEND
    df_bronze.write.format("delta").mode("append").saveAsTable("workspace.crypto_analytics.bronze_crypto")
    
    print("✓ Sucesso! Dados em tempo real coletados e acumulados na tabela bronze_crypto.")
else:
    raise Exception(f"❌ Falha na comunicação com a API. Código de status HTTP: {response.status_code}")